Test of morris like approach applied by kratzert 2019

In [43]:
from pathlib import Path
from neuralhydrology.modelzoo import EALSTM

from neuralhydrology.datasetzoo import get_dataset, camelsus
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.cudalstm import CudaLSTM
from neuralhydrology.modelzoo.customlstm import CustomLSTM
from neuralhydrology.nh_run import start_run
from neuralhydrology.utils.config import Config

import numpy as np
import torch

from torch.utils.data import DataLoader

from tqdm import tqdm
import pandas as pd

In [ ]:
"""
This file is part of the accompanying code to our manuscript:

Kratzert, F., Klotz, D., Shalev, G., Klambauer, G., Hochreiter, S., Nearing, G., "Benchmarking
a Catchment-Aware Long Short-Term Memory Network (LSTM) for Large-Scale Hydrological Modeling".
submitted to Hydrol. Earth Syst. Sci. Discussions (2019)

You should have received a copy of the Apache-2.0 license along with the code. If not,
see <https://opensource.org/licenses/Apache-2.0>
"""

import numpy as np
import torch

DEVICE = torch.device("cuda:0")


def get_morris_gradient(model: torch.nn.Module,
						loader: torch.utils.data.DataLoader) -> torch.Tensor:
	"""Calculate gradients w.r.t static network inputs.

	TODO: Update Docstring with ref to paper
	
		Parameters
	----------
	model : nn.Module
		The PyTorch model to train
	loader : DataLoader
		PyTorch DataLoader containing the basin data in batches.
	
	Returns
	-------
	torch.Tensor
		[description]
	"""
	model.eval()
	grads = []

	# adapt for EA-LSTM in NH
	for batch in loader:
		
		batch["x_s"] = torch.autograd.Variable(batch["x_s"], requires_grad=True)

		model.zero_grad()
		
		pred = model(batch)
		
		# create mask to not compute gradients for days, where no discharge data exists, since these days are not in the set
		mask = ~torch.isnan(batch["y"])
			
		grad = torch.autograd.grad(pred["y_hat"],
								   batch["x_s"],
								   grad_outputs=mask.float(),
								   create_graph=False)
		grads.append(grad[0][:,:].detach().cpu().numpy())
		print(torch.isnan(grad[0][:,:]).sum())
		print(len(grad[0]))

	return np.concatenate(grads, axis=0), grad

In [45]:
# load model version 

run_dir_path = Path("/home/wuhlmann/BA/test_runs/runs/full_q_512_3011_185525")
cfg = Config(run_dir_path/"config.yml")

model = EALSTM(cfg=cfg)
scaler = load_scaler(run_dir=run_dir_path)

model_weights = torch.load("/home/wuhlmann/BA/test_runs/runs/full_q_512_3011_185525/model_epoch030.pt", map_location="cuda:0")
model.load_state_dict(model_weights)



<All keys matched successfully>

In [46]:
with open("/home/wuhlmann/BA/data/processed_data/test_splits/test_basin_ids.txt", "r") as test_basin_file: 
	
	basins = list(map(lambda x: x[:-1], test_basin_file.readlines()))


In [47]:
feature_ranking = {}

for basin in range(1):

	ds_test = get_dataset(cfg=cfg, is_train=False, period="train", scaler=scaler, basin="55")
	loader = DataLoader(ds_test, batch_size=1024, shuffle=False, num_workers=0, collate_fn=ds_test.collate_fn)

	gradients, grad = get_morris_gradient(model, loader)

	mean_abs_gradient = np.nanmean(np.abs(gradients), axis=0)

	# convert to pandas Series
	data = {}
	for name, value in zip(list(cfg.static_attributes), mean_abs_gradient):
		data[name] = value
	feature_ranking["2"] = pd.Series(data=data)

tensor(10556)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
1024
tensor(0)
202


In [48]:
feature_ranking["2"]

area_calc     1508.622681
elev_mean     2884.089355
elev_ran       792.467834
slope_mean     760.837769
p_mean        1778.821899
et0_mean      2032.729004
arid_1        1029.418945
p_season      1263.029297
hi_prec_fr     918.660767
hi_prec_du     801.261353
lo_prec_fr    1009.038696
lo_prec_du     553.901733
frac_snow     1356.561035
agr_fra       2204.998047
bare_fra      1633.030884
forest_fra     926.825317
glac_fra       964.568848
lake_fra       434.065063
urban_fra     4012.041748
lai_max       1290.551880
lai_diff      1097.771729
gvf_max       1836.895996
gvf_diff       887.586670
bedrk_dep     1315.998535
soil_poros    1089.212646
soil_condu     747.387207
sand_fra      2120.609131
silt_fra      1232.545654
clay_fra       843.899902
dtype: float32

In [49]:
normalized_ranks = {}
for basin, ranks in feature_ranking.items():
    ranks = (ranks - ranks.min()) / (ranks.max() - ranks.min())
    normalized_ranks[basin] = ranks
df = pd.DataFrame(data={basin: ranks for basin, ranks in normalized_ranks.items()})
df.mean(axis=1).sort_values(ascending=False)

urban_fra     1.000000
elev_mean     0.684751
agr_fra       0.494954
sand_fra      0.471368
et0_mean      0.446807
gvf_max       0.392074
p_mean        0.375843
bare_fra      0.335096
area_calc     0.300326
frac_snow     0.257826
bedrk_dep     0.246489
lai_max       0.239377
p_season      0.231685
silt_fra      0.223165
lai_diff      0.185498
soil_poros    0.183106
arid_1        0.166394
lo_prec_fr    0.160698
glac_fra      0.148269
forest_fra    0.137720
hi_prec_fr    0.135438
gvf_diff      0.126754
clay_fra      0.114544
hi_prec_du    0.102627
elev_ran      0.100169
slope_mean    0.091329
soil_condu    0.087570
lo_prec_du    0.033493
lake_fra      0.000000
dtype: float32